# Market Regime Detection for Portfolio Allocation with RL + Explainability

This notebook builds a regime-aware portfolio allocation pipeline and then explains policy behavior with temporal attention-style diagnostics.

## What This Notebook Covers
1. Regime detection with Gaussian HMM
2. RL training with DQN, A2C, and PPO
3. Test-period evaluation and policy behavior diagnostics
4. Explainability views:
   - DQN surrogate attention + gradient saliency
   - Finance dashboard linking returns, actions, and regimes
   - PPO post-hoc surrogate attention + policy saliency

## Important Clarification
- DQN section uses value-based learning and is explained with surrogate attention + saliency.
- PPO section in this notebook uses `MlpPolicy` (no native attention layer in PPO policy).
- PPO attention plots are post-hoc explainability, not internal PPO attention weights.

## Execution Guide (Recommended Order)

1. Run Sections 1 to 5 to prepare data and environments.
2. Choose training path:
   - DQN path: Section 6 + Sections 7 to 9.2
   - PPO path: PPO training cell + Section 9.3
3. Run Section 10 only when comparing against the baseline setup.

This order keeps variables aligned and avoids stale-state confusion in long notebook sessions.

## 1. Install and Import Dependencies

In [11]:
import os
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
if not (repo_root / "ml").exists():
    found_root = None
    for candidate in [repo_root] + list(repo_root.parents):
        if (candidate / "ml").exists() and (candidate / "data").exists():
            found_root = candidate
            break
    if found_root is None:
        raise FileNotFoundError("Could not locate project root containing 'ml' and 'data'.")
    repo_root = found_root

sys.path.insert(0, str(repo_root))

def _rel_display(path_obj: Path) -> str:
    try:
        return str(path_obj.relative_to(repo_root))
    except ValueError:
        return str(path_obj)

import numpy as np
import pandas as pd
import seaborn as sns
import warnings

# FinRL (stable-baselines3)
from stable_baselines3 import DQN, A2C, PPO
from stable_baselines3.common.vec_env import DummyVecEnv

# ML libraries
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Project modules
from ml.models import GaussianHMMRegimeDetector
from ml.environments import WeeklyPortfolioEnv
from ml.training_utils import train_dqn_finrl, evaluate_episode
from ml import load_hyperparameter_config

# Visualization
from tqdm import tqdm

warnings.filterwarnings('ignore')

# Load centralized hyperparameters
hp_bundle = load_hyperparameter_config()
hp_cfg = hp_bundle.values
FAST_MODE = hp_bundle.fast_mode

env_cfg = hp_cfg['environment']
dqn_cfg = hp_cfg['dqn']
a2c_cfg = hp_cfg['a2c']
ppo_cfg = hp_cfg['ppo']
ppo_attn_cfg = hp_cfg['ppo_native_attention']
hpo_cfg = hp_cfg['hpo']
SEQ_LEN = hp_cfg['general']['sequence_length']

print(f"Hyperparameter mode: {'FAST' if FAST_MODE else 'FULL'}")
print(f"Configured sequence length: {SEQ_LEN}")

# Saved model artifacts for split-notebook workflow
model_dir = repo_root / 'output' / 'models'
MODEL_PATHS = {
    'dqn': model_dir / 'dqn_finrl_agent.zip',
    'a2c': model_dir / 'a2c_finrl_agent.zip',
    'ppo': model_dir / 'ppo_finrl_agent.zip',
    'ppo_native_attention': model_dir / 'ppo_native_attention_finrl_agent.zip',
}
print(f"Model artifacts directory: {_rel_display(model_dir)}")

# Set random seeds for reproducibility
np.random.seed(42)
import torch
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
elif torch.backends.mps.is_available():
    torch.mps.manual_seed(42)

# Device configuration (order: CUDA > MPS > CPU)
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device: {device}")

sns.set_palette("husl")

Hyperparameter mode: FULL
Configured sequence length: 4
Model artifacts directory: output/models
Using device: cuda


## 2. Load and Prepare Data from Pipeline

In [12]:
# Load processed data from the data pipeline
# Use repo_root so this works regardless of current working directory.
data_path = repo_root / 'data' / 'processed'

# Load weekly features (market + macro)
market_features = pd.read_csv(data_path / 'market_features_weekly.csv', index_col=0, parse_dates=True)
macro_features = pd.read_csv(data_path / 'macro_features_weekly.csv', index_col=0, parse_dates=True)
asset_targets = pd.read_csv(data_path / 'weekly_asset_targets.csv', index_col=0, parse_dates=True)

print("Data path: data/processed")
print(f"Market features shape: {market_features.shape}")
print(f"Macro features shape: {macro_features.shape}")
print(f"Asset targets shape: {asset_targets.shape}")

# Align indices
common_index = market_features.index.intersection(macro_features.index).intersection(asset_targets.index)
market_features = market_features.loc[common_index].sort_index()
macro_features = macro_features.loc[common_index].sort_index()
asset_targets = asset_targets.loc[common_index].sort_index()

print(f"\nAligned data shape: {market_features.shape}")
print(f"Date range: {market_features.index[0]} to {market_features.index[-1]}")

# Combine features for regime detection
all_features = pd.concat([market_features, macro_features], axis=1)
print(f"Combined features for HMM: {all_features.shape}")
print(f"Features: {list(all_features.columns)[:10]}...")

Data path: data/processed
Market features shape: (638, 37)
Macro features shape: (638, 33)
Asset targets shape: (638, 8)

Aligned data shape: (638, 37)
Date range: 2014-01-03 00:00:00 to 2026-03-20 00:00:00
Combined features for HMM: (638, 70)
Features: ['spy_ret_1d', 'spy_ret_5d', 'spy_ret_20d', 'spy_vol_5d', 'spy_vol_20d', 'spy_drawdown_60d', 'spy_ma_gap_5_20', 'spy_intraday_range', 'spy_volume_z_20', 'tlt_ret_1d']...


## 3. Gaussian HMM Regime Detection

In [13]:
# Split into Train/Validation/Test (2014-2020 / 2021-2022 / 2023-2026)
split_date_train = '2020-12-31'
split_date_val = '2022-12-31'

train_mask = all_features.index <= split_date_train
val_mask = (all_features.index > split_date_train) & (all_features.index <= split_date_val)
test_mask = all_features.index > split_date_val

train_features = all_features[train_mask].reset_index(drop=True)
train_returns = asset_targets[train_mask]

val_features = all_features[val_mask].reset_index(drop=True)
val_returns = asset_targets[val_mask]

test_features = all_features[test_mask].reset_index(drop=True)
test_returns = asset_targets[test_mask]

# Select only numeric columns and drop NaN
numeric_cols = train_features.select_dtypes(include=[np.number]).columns
train_features = train_features[numeric_cols].fillna(0)
val_features = val_features[numeric_cols].fillna(0)
test_features = test_features[numeric_cols].fillna(0)

print(f"Train data: {len(train_features)} weeks")
print(f"Val data: {len(val_features)} weeks")
print(f"Test data: {len(test_features)} weeks")
print(f"Train features shape: {train_features.shape} (numeric features only)")

# Fit HMM on training data
print("\n=== Fit Gaussian HMM ===")
hmm = GaussianHMMRegimeDetector(n_regimes=4, pca_components=10, random_state=42)
hmm.fit(train_features, regime_names=["Risk-On", "Neutral", "Defensive", "Panic"])

print(f"HMM fitted with {hmm.n_regimes} regimes")
print(f"Regime names: {hmm.get_regime_names()}")

# Predict regimes for all periods
train_regime_posteriors = hmm.predict_proba(train_features)
val_regime_posteriors = hmm.predict_proba(val_features)
test_regime_posteriors = hmm.predict_proba(test_features)

print(f"\nRegime posteriors shape:")
print(f"  Train: {train_regime_posteriors.shape}")
print(f"  Val: {val_regime_posteriors.shape}")
print(f"  Test: {test_regime_posteriors.shape}")

# Visualize regime detection with Plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots

predicted_regimes = hmm.predict_regimes(train_features)
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=False,
    vertical_spacing=0.12,
    subplot_titles=(
        'Market Regimes (Train Period) - HMM Posterior Probabilities',
        'Most Likely Market Regime Over Time',
    ),
)

for regime_id in range(hmm.n_regimes):
    fig.add_trace(
        go.Scatter(
            x=list(range(len(train_regime_posteriors))),
            y=train_regime_posteriors[:, regime_id],
            mode='lines',
            name=hmm.regime_names[regime_id],
        ),
        row=1,
        col=1,
    )

fig.add_trace(
    go.Scatter(
        x=list(range(len(predicted_regimes))),
        y=predicted_regimes,
        mode='markers',
        marker=dict(size=6, opacity=0.7),
        name='Predicted Regime',
        showlegend=False,
    ),
    row=2,
    col=1,
)

fig.update_xaxes(title_text='Week', row=1, col=1)
fig.update_yaxes(title_text='Probability', row=1, col=1)
fig.update_xaxes(title_text='Week', row=2, col=1)
fig.update_yaxes(
    title_text='Regime ID',
    tickmode='array',
    tickvals=list(range(hmm.n_regimes)),
    ticktext=hmm.regime_names,
    row=2,
    col=1,
)
fig.update_layout(height=820, width=1250, legend=dict(orientation='h', y=1.02, x=0))
fig.show()

print("\nRegime Statistics (Train Period):")
regime_stats = hmm.get_regime_stats()
for regime_id, stats in regime_stats.items():
    print(f"\n{hmm.regime_names[regime_id]}:")
    for feature, value in list(stats.items())[:5]:
        print(f"  {feature}: {value:.4f}")

Train data: 365 weeks
Val data: 105 weeks
Test data: 168 weeks
Train features shape: (365, 66) (numeric features only)

=== Fit Gaussian HMM ===
HMM fitted with 4 regimes
Regime names: ['Risk-On', 'Neutral', 'Defensive', 'Panic']

Regime posteriors shape:
  Train: (365, 4)
  Val: (105, 4)
  Test: (168, 4)



Regime Statistics (Train Period):

Risk-On:
  spy_ret_1d: -0.0003
  spy_ret_5d: 0.0020
  spy_ret_20d: 0.0085
  spy_vol_5d: 0.0076
  spy_vol_20d: 0.0081

Neutral:
  spy_ret_1d: 0.0006
  spy_ret_5d: 0.0029
  spy_ret_20d: 0.0114
  spy_vol_5d: 0.0064
  spy_vol_20d: 0.0070

Defensive:
  spy_ret_1d: 0.0031
  spy_ret_5d: -0.0192
  spy_ret_20d: -0.0840
  spy_vol_5d: 0.0399
  spy_vol_20d: 0.0407

Panic:
  spy_ret_1d: 0.0019
  spy_ret_5d: 0.0100
  spy_ret_20d: 0.0471
  spy_vol_5d: 0.0127
  spy_vol_20d: 0.0150


## 4. Create Training, Validation, and Test Environments

In [14]:
# Asset returns for portfolio: [SPY, TLT, GLD, Cash(~0)]
# Check available columns and use those
print(f"Available columns in asset_targets: {asset_targets.columns.tolist()}")

# Use columns that match the expected assets
spy_col = 'SPY_return' if 'SPY_return' in asset_targets.columns else next((c for c in asset_targets.columns if 'SPY' in c.upper()), None)
tlt_col = 'TLT_return' if 'TLT_return' in asset_targets.columns else next((c for c in asset_targets.columns if 'TLT' in c.upper()), None)
gld_col = 'GLD_return' if 'GLD_return' in asset_targets.columns else next((c for c in asset_targets.columns if 'GLD' in c.upper()), None)

print(f"Using columns: SPY={spy_col}, TLT={tlt_col}, GLD={gld_col}")

spy_returns = asset_targets[spy_col] if spy_col else np.random.randn(len(asset_targets)) * 0.02
tlt_returns = asset_targets[tlt_col] if tlt_col else np.random.randn(len(asset_targets)) * 0.01
gld_returns = asset_targets[gld_col] if gld_col else np.random.randn(len(asset_targets)) * 0.015
cash_returns = np.zeros_like(spy_returns)  # Cash has ~0 return

portfolio_returns = pd.DataFrame({
    'SPY': spy_returns.values if hasattr(spy_returns, 'values') else spy_returns,
    'TLT': tlt_returns.values if hasattr(tlt_returns, 'values') else tlt_returns,
    'GLD': gld_returns.values if hasattr(gld_returns, 'values') else gld_returns,
    'Cash': cash_returns,
})

# Normalize features (important for neural network)
scaler = StandardScaler()
train_market_scaled = scaler.fit_transform(train_features)
val_market_scaled = scaler.transform(val_features)
test_market_scaled = scaler.transform(test_features)

train_features_normalized = train_features.copy()
train_features_normalized[:] = train_market_scaled

val_features_normalized = val_features.copy()
val_features_normalized[:] = val_market_scaled

test_features_normalized = test_features.copy()
test_features_normalized[:] = test_market_scaled

# Create environments from centralized config
train_env = WeeklyPortfolioEnv(
    features=train_features_normalized,
    regime_posteriors=train_regime_posteriors,
    asset_returns=portfolio_returns.iloc[:len(train_features_normalized)],
    transaction_cost=env_cfg['transaction_cost'],
    volatility_penalty=env_cfg['volatility_penalty'],
    lookback_vol=env_cfg['lookback_vol'],
    seq_len=SEQ_LEN
)

val_env = WeeklyPortfolioEnv(
    features=val_features_normalized,
    regime_posteriors=val_regime_posteriors,
    asset_returns=portfolio_returns.iloc[len(train_features_normalized):len(train_features_normalized)+len(val_features_normalized)],
    transaction_cost=env_cfg['transaction_cost'],
    volatility_penalty=env_cfg['volatility_penalty'],
    lookback_vol=env_cfg['lookback_vol'],
    seq_len=SEQ_LEN
)

test_env = WeeklyPortfolioEnv(
    features=test_features_normalized,
    regime_posteriors=test_regime_posteriors,
    asset_returns=portfolio_returns.iloc[len(train_features_normalized)+len(val_features_normalized):],
    transaction_cost=env_cfg['transaction_cost'],
    volatility_penalty=env_cfg['volatility_penalty'],
    lookback_vol=env_cfg['lookback_vol'],
    seq_len=SEQ_LEN
)

print(f"Environment Configuration:")
print(f"  State dimension: {train_env.observation_space.shape}")
print(f"  Action dimension: {train_env.action_space.n}")
print(f"  Sequence length: {train_env.seq_len}")
print(f"  Portfolio templates: {len(train_env.PORTFOLIO_TEMPLATES)}")
print(f"\nPortfolio Actions:")
for action_id, name in enumerate(train_env.ACTION_NAMES):
    print(f"  {action_id}: {name}")

Available columns in asset_targets: ['spy_weekly_close', 'tlt_weekly_close', 'gld_weekly_close', 'week_last_trade_date', 'next_return_spy', 'next_return_tlt', 'next_return_gld', 'source']
Using columns: SPY=spy_weekly_close, TLT=tlt_weekly_close, GLD=gld_weekly_close
Environment Configuration:
  State dimension: (4, 74)
  Action dimension: 7
  Sequence length: 4
  Portfolio templates: 7

Portfolio Actions:
  0: cash_only
  1: spy_only
  2: tlt_only
  3: gld_only
  4: spy_80_tlt_20
  5: balanced_60_30_10
  6: defensive_20_60_20


## 5. Initialize Attention DQN Agent

In [15]:
# Initialize DQN agent using centralized hyperparameter config
agent = DQN(
    'MlpPolicy',
    train_env,
    learning_rate=dqn_cfg['learning_rate'],
    buffer_size=dqn_cfg['buffer_size'],
    learning_starts=dqn_cfg['learning_starts'],
    batch_size=dqn_cfg['batch_size'],
    tau=1.0,
    gamma=dqn_cfg['gamma'],
    train_freq=1,
    target_update_interval=dqn_cfg['target_update_interval'],
    exploration_fraction=dqn_cfg['exploration_fraction'],
    exploration_initial_eps=dqn_cfg['exploration_initial_eps'],
    exploration_final_eps=dqn_cfg['exploration_final_eps'],
    device=device,
    verbose=1
)

print("=== FinRL DQN Agent Initialized ===")
print(f"Device: {agent.device}")
print(f"Policy: MlpPolicy")
print(f"State space: {train_env.observation_space}")
print(f"Action space: {train_env.action_space}")
print(f"Learning rate: {agent.learning_rate}")
print(f"Gamma: {agent.gamma}")
print(f"\nBuffer size: {agent.buffer_size}")
print(f"Batch size: {agent.batch_size}")
print(f"Target update interval: {agent.target_update_interval}")

# Test forward pass (gymnasium returns (obs, info) tuple)
obs, info = train_env.reset()
print(f"\nObservation shape: {obs.shape}")
print(f"Info keys: {info.keys() if isinstance(info, dict) else 'N/A'}")

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
=== FinRL DQN Agent Initialized ===
Device: cuda
Policy: MlpPolicy
State space: Box(-inf, inf, (4, 74), float32)
Action space: Discrete(7)
Learning rate: 0.0001
Gamma: 0.99

Buffer size: 250000
Batch size: 256
Target update interval: 2000

Observation shape: (4, 74)
Info keys: dict_keys([])


## 6. Train Attention-Enhanced DQN Agent

In [16]:
# Load saved DQN weights and evaluate on test period
from stable_baselines3 import DQN

dqn_model_path = MODEL_PATHS['dqn']
if not dqn_model_path.exists():
    raise FileNotFoundError(
        f"Missing DQN weights at {dqn_model_path}. Run notebooks_split/01_training_and_tuning.ipynb first."
    )

agent = DQN.load(str(dqn_model_path), env=test_env, device=device)
print(f"Loaded DQN weights from: {dqn_model_path}")

test_eval = evaluate_episode(agent, test_env, deterministic=True)
actions_df = pd.DataFrame(test_eval.get('actions', []))

print("DQN Test Performance:")
print(f"  Total Reward: {test_eval['reward']:.4f}")
print(f"  Cumulative Return: {test_eval.get('cumulative_return', 0):.4f}")
print(f"  Sharpe Ratio: {test_eval.get('sharpe_ratio', 0):.4f}")
print(f"  Max Drawdown: {test_eval.get('max_drawdown', 0):.4f}")

Loaded DQN weights from: /home/yuaylong/Market-Regime-Detection-for-RL-Allocation/output/models/dqn_finrl_agent.zip
DQN Test Performance:
  Total Reward: 86325.3338
  Cumulative Return: inf
  Sharpe Ratio: 41.4247
  Max Drawdown: nan


In [17]:
# Load saved PPO and PPO Native Attention weights for cross-model comparison
from stable_baselines3 import PPO

ppo_model_path = MODEL_PATHS['ppo']
if ppo_model_path.exists():
    ppo_agent = PPO.load(str(ppo_model_path), env=test_env, device=device)
    ppo_test_eval = evaluate_episode(ppo_agent, test_env, deterministic=True)
    ppo_prob_df = pd.DataFrame(ppo_test_eval.get('actions', []))
    print(f"Loaded PPO weights from: {ppo_model_path}")
else:
    ppo_agent, ppo_test_eval, ppo_prob_df = None, None, pd.DataFrame()
    print(f"WARNING: PPO weights not found at {ppo_model_path}")

ppo_attn_model_path = MODEL_PATHS['ppo_native_attention']
if ppo_attn_model_path.exists():
    try:
        ppo_attn_agent = PPO.load(str(ppo_attn_model_path), env=test_env, device=device)
        ppo_attn_test_eval = evaluate_episode(ppo_attn_agent, test_env, deterministic=True)
        attn_prob_df = pd.DataFrame(ppo_attn_test_eval.get('actions', []))
        print(f"Loaded PPO Native Attention weights from: {ppo_attn_model_path}")
    except Exception as exc:
        ppo_attn_agent, ppo_attn_test_eval, attn_prob_df = None, None, pd.DataFrame()
        print(f"WARNING: Could not load PPO Native Attention weights ({exc})")
else:
    ppo_attn_agent, ppo_attn_test_eval, attn_prob_df = None, None, pd.DataFrame()
    print(f"WARNING: PPO Native Attention weights not found at {ppo_attn_model_path}")

Loaded PPO weights from: /home/yuaylong/Market-Regime-Detection-for-RL-Allocation/output/models/ppo_finrl_agent.zip
Loaded PPO Native Attention weights from: /home/yuaylong/Market-Regime-Detection-for-RL-Allocation/output/models/ppo_native_attention_finrl_agent.zip


In [18]:
# Compare DQN, PPO (MLP), and PPO (Native Attention) in one panel
print("=== Model Comparison: DQN vs PPO-MLP vs PPO-Native-Attention ===")

import plotly.express as px
import plotly.graph_objects as go

def _safe_metric(eval_dict, key):
    return float(eval_dict.get(key, np.nan)) if isinstance(eval_dict, dict) else np.nan

def _action_entropy_from_series(action_series):
    if action_series is None or len(action_series) == 0:
        return np.nan
    probs = action_series.value_counts(normalize=True).values
    return float(-(probs * np.log(probs + 1e-12)).sum())

def _build_model_row(model_name, eval_dict=None, action_df=None, action_col=None):
    entropy = np.nan
    if isinstance(action_df, pd.DataFrame) and action_col in action_df.columns:
        entropy = _action_entropy_from_series(action_df[action_col])
    elif isinstance(eval_dict, dict) and 'actions' in eval_dict:
        tmp_df = pd.DataFrame(eval_dict['actions'])
        if 'action_name' in tmp_df.columns:
            entropy = _action_entropy_from_series(tmp_df['action_name'])

    return {
        'model': model_name,
        'reward': _safe_metric(eval_dict, 'reward'),
        'cumulative_return': _safe_metric(eval_dict, 'cumulative_return'),
        'sharpe_ratio': _safe_metric(eval_dict, 'sharpe_ratio'),
        'max_drawdown': _safe_metric(eval_dict, 'max_drawdown'),
        'action_entropy': entropy,
    }

rows = [
    _build_model_row('DQN', globals().get('test_eval'), globals().get('actions_df'), 'action_name'),
    _build_model_row('PPO-MLP', globals().get('ppo_test_eval'), globals().get('ppo_prob_df'), 'chosen_action'),
    _build_model_row('PPO-Native-Attn', globals().get('ppo_attn_test_eval'), globals().get('attn_prob_df'), 'chosen_action'),
]

cmp_df = pd.DataFrame(rows)
print(cmp_df.round(4))

metric_cols = ['reward', 'cumulative_return', 'sharpe_ratio', 'max_drawdown', 'action_entropy']
if cmp_df[metric_cols].isna().all(axis=None):
    raise RuntimeError(
        "No model results found. Run: DQN eval (Section 8), PPO-MLP (Section 6.2), and/or PPO-Native-Attn (Section 6.3)."
    )

perf_long = cmp_df.melt(
    id_vars='model',
    value_vars=['reward', 'cumulative_return', 'sharpe_ratio', 'max_drawdown'],
    var_name='metric',
    value_name='value',
)

fig_perf = px.bar(
    perf_long,
    x='metric',
    y='value',
    color='model',
    barmode='group',
    title='Performance Comparison: DQN vs PPO Variants',
    labels={'metric': 'Metric', 'value': 'Value', 'model': 'Model'},
)
fig_perf.update_layout(height=520, width=1280, legend=dict(orientation='h', y=1.02, x=0))
fig_perf.show()

fig_entropy = px.bar(
    cmp_df,
    x='model',
    y='action_entropy',
    color='model',
    title='Action Diversity (Entropy) Comparison',
    labels={'action_entropy': 'Action entropy', 'model': 'Model'},
)
fig_entropy.update_layout(height=420, width=900, showlegend=False)
fig_entropy.show()

radar_metrics = ['reward', 'cumulative_return', 'sharpe_ratio', 'action_entropy']
radar_df = cmp_df[['model'] + radar_metrics].copy()

for metric in radar_metrics:
    col = radar_df[metric].astype(float)
    valid = col.dropna()
    if len(valid) == 0:
        radar_df[metric] = np.nan
    elif abs(valid.max() - valid.min()) < 1e-12:
        radar_df[metric] = 0.5
    else:
        radar_df[metric] = (col - valid.min()) / (valid.max() - valid.min())

fig_radar = go.Figure()
for _, row in radar_df.iterrows():
    vals = [row[m] for m in radar_metrics]
    fig_radar.add_trace(
        go.Scatterpolar(
            r=vals + [vals[0]],
            theta=radar_metrics + [radar_metrics[0]],
            fill='toself',
            name=row['model'],
        )
    )

fig_radar.update_layout(
    title='Normalized Multi-Metric Comparison (higher is better)',
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    height=560,
    width=900,
    legend=dict(orientation='h', y=1.05, x=0),
)
fig_radar.show()

print("Comparison complete.")
print("Tip: if PPO-Native-Attn row is NaN, run Section 6.3 first.")

=== Model Comparison: DQN vs PPO-MLP vs PPO-Native-Attention ===
             model      reward  cumulative_return  sharpe_ratio  max_drawdown  \
0              DQN  86325.3338                inf       41.4247           NaN   
1          PPO-MLP  85742.2416                inf       41.0782           NaN   
2  PPO-Native-Attn  86325.3338                inf       41.4247           NaN   

   action_entropy  
0         -0.0000  
1          0.1592  
2         -0.0000  


Comparison complete.
Tip: if PPO-Native-Attn row is NaN, run Section 6.3 first.


In [19]:
# Auto-rank models with a simple risk-adjusted composite score
print("=== Auto Model Scoring and Winner Selection ===")

import numpy as np
import pandas as pd

def _safe_float(d, k, default=np.nan):
    try:
        return float(d.get(k, default)) if isinstance(d, dict) else float(default)
    except Exception:
        return float(default)

def _finite_or_default(x, default=0.0):
    try:
        x = float(x)
    except Exception:
        return float(default)
    return x if np.isfinite(x) else float(default)

def _calc_action_entropy(action_series):
    if action_series is None or len(action_series) == 0:
        return np.nan
    probs = action_series.value_counts(normalize=True).values
    return float(-(probs * np.log(probs + 1e-12)).sum())

def _estimate_max_drawdown_from_actions(action_df):
    if not isinstance(action_df, pd.DataFrame) or 'return' not in action_df.columns or len(action_df) == 0:
        return np.nan

    r = pd.to_numeric(action_df['return'], errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
    if r.empty:
        return np.nan

    # Build drawdown in log space to avoid overflow for long/high-growth series.
    # Clip only invalid downside (< -100%) to keep log1p stable.
    r = np.maximum(r.to_numpy(dtype=float), -0.999999)
    log_equity = np.cumsum(np.log1p(r))
    run_max = np.maximum.accumulate(log_equity)
    drawdowns = np.exp(log_equity - run_max) - 1.0
    return float(np.min(drawdowns)) if len(drawdowns) else np.nan

# Auto-backfill DQN test evaluation if DQN was loaded but not evaluated in this session.
if ('test_eval' not in globals() or not isinstance(globals().get('test_eval'), dict)) and ('agent' in globals()) and ('test_env' in globals()):
    try:
        test_eval = evaluate_episode(agent, test_env, deterministic=True)
        if isinstance(test_eval, dict) and 'actions' in test_eval:
            actions_df = pd.DataFrame(test_eval['actions'])
        print("DQN eval auto-generated from current agent/test_env for ranking.")
    except Exception as e:
        print(f"DQN auto-eval skipped: {e}")

def _build_row(name, eval_dict=None, action_df=None, action_col=None):
    entropy = np.nan
    action_df_for_dd = action_df if isinstance(action_df, pd.DataFrame) else None

    if isinstance(action_df, pd.DataFrame) and action_col in action_df.columns:
        entropy = _calc_action_entropy(action_df[action_col])

    eval_actions_df = None
    if isinstance(eval_dict, dict) and 'actions' in eval_dict:
        eval_actions_df = pd.DataFrame(eval_dict['actions'])
        if pd.isna(entropy) and 'action_name' in eval_actions_df.columns:
            entropy = _calc_action_entropy(eval_actions_df['action_name'])
        if action_df_for_dd is None or 'return' not in action_df_for_dd.columns:
            action_df_for_dd = eval_actions_df

    reward_raw = _safe_float(eval_dict, 'reward', np.nan)
    cumulative_return_raw = _safe_float(eval_dict, 'cumulative_return', np.nan)
    sharpe_ratio_raw = _safe_float(eval_dict, 'sharpe_ratio', np.nan)
    max_drawdown_raw = _safe_float(eval_dict, 'max_drawdown', np.nan)

    # Fallback: derive max drawdown from action-level returns when env stats are missing.
    if not np.isfinite(max_drawdown_raw):
        dd_fallback = _estimate_max_drawdown_from_actions(action_df_for_dd)
        if np.isfinite(dd_fallback):
            max_drawdown_raw = dd_fallback

    reward_score = _finite_or_default(reward_raw, 0.0)
    cumulative_return_score = _finite_or_default(cumulative_return_raw, 0.0)
    sharpe_ratio_score = _finite_or_default(sharpe_ratio_raw, 0.0)
    max_drawdown_score = _finite_or_default(max_drawdown_raw, 0.0)

    has_eval = isinstance(eval_dict, dict) and len(eval_dict) > 0
    composite_score = np.nan
    if has_eval:
        # Composite score (higher is better):
        #   + sharpe_ratio
        #   + 0.5 * cumulative_return
        #   - 0.5 * abs(max_drawdown)
        composite_score = sharpe_ratio_score + 0.5 * cumulative_return_score - 0.5 * abs(max_drawdown_score)

    return {
        'model': name,
        'has_eval': has_eval,
        'reward': reward_raw,
        'cumulative_return': cumulative_return_raw,
        'sharpe_ratio': sharpe_ratio_raw,
        'max_drawdown': max_drawdown_raw,
        'action_entropy': entropy,
        'composite_score': composite_score,
    }

rows = [
    _build_row('DQN', globals().get('test_eval'), globals().get('actions_df'), 'action_name'),
    _build_row('PPO-MLP', globals().get('ppo_test_eval'), globals().get('ppo_prob_df'), 'chosen_action'),
    _build_row('PPO-Native-Attn', globals().get('ppo_attn_test_eval'), globals().get('attn_prob_df'), 'chosen_action'),
]

score_df = pd.DataFrame(rows)
print("\nModel availability:")
print(score_df[['model', 'has_eval', 'reward', 'cumulative_return', 'sharpe_ratio', 'max_drawdown']].round(4))

rank_df = score_df.dropna(subset=['composite_score']).copy()
rank_df = rank_df[np.isfinite(rank_df['composite_score'])]

if rank_df.empty:
    raise RuntimeError(
        "No evaluated model found in kernel state. Run at least one test-evaluation cell first."
    )

rank_df = rank_df.sort_values('composite_score', ascending=False).reset_index(drop=True)
winner = rank_df.iloc[0]

print("\nRanking (best to worst):")
print(rank_df[['model', 'composite_score', 'sharpe_ratio', 'cumulative_return', 'max_drawdown', 'action_entropy']].round(4))

print("\nWinner:")
print(f"  Model: {winner['model']}")
print(f"  Composite score: {winner['composite_score']:.4f}")
print(f"  Sharpe: {winner['sharpe_ratio'] if pd.notna(winner['sharpe_ratio']) else float('nan'):.4f} | CumReturn: {winner['cumulative_return'] if pd.notna(winner['cumulative_return']) else float('nan'):.4f} | MaxDD: {winner['max_drawdown'] if pd.notna(winner['max_drawdown']) else float('nan'):.4f}")

print("\nDecision note:")
print("- Prefer the top composite score, then choose the lower drawdown model if scores are close.")

=== Auto Model Scoring and Winner Selection ===

Model availability:
             model  has_eval      reward  cumulative_return  sharpe_ratio  \
0              DQN      True  86325.3338                inf       41.4247   
1          PPO-MLP      True  85742.2416                inf       41.0782   
2  PPO-Native-Attn      True  86325.3338                inf       41.4247   

   max_drawdown  
0           0.0  
1           0.0  
2           0.0  

Ranking (best to worst):
             model  composite_score  sharpe_ratio  cumulative_return  \
0              DQN          41.4247       41.4247                inf   
1  PPO-Native-Attn          41.4247       41.4247                inf   
2          PPO-MLP          41.0782       41.0782                inf   

   max_drawdown  action_entropy  
0           0.0         -0.0000  
1           0.0         -0.0000  
2           0.0          0.1592  

Winner:
  Model: DQN
  Composite score: 41.4247
  Sharpe: 41.4247 | CumReturn: inf | MaxDD: 0.0000

## 6.5 Hyperparameter Fine-Tune (PPO Native Attention)

Compact random/grid search over key PPO-native-attention hyperparameters using validation reward.
The best configuration is retrained and stored as `ppo_attn_hpo_best`.

In [20]:
# Hyperparameter fine-tune for PPO Native Attention (config-driven search)
print("=== Hyperparameter Fine-Tuning: PPO Native Attention ===")

import pandas as pd
import plotly.express as px

if 'TemporalAttentionExtractor' not in globals():
    raise RuntimeError("Run Section 6.3 first so TemporalAttentionExtractor is defined.")

print(f"FAST_MODE={FAST_MODE}")

selected_trials = hpo_cfg['selected_trials']
hpo_timesteps = hpo_cfg['train_timesteps']
hpo_rows = []
best_score = -1e18
best_cfg = None
best_agent = None

for trial_idx, cfg in enumerate(selected_trials, start=1):
    print(f"\nTrial {trial_idx}/{len(selected_trials)}: {cfg}")

    policy_kwargs_hpo = dict(
        features_extractor_class=TemporalAttentionExtractor,
        features_extractor_kwargs=dict(features_dim=ppo_attn_cfg['features_dim']),
        net_arch=ppo_attn_cfg['net_arch'],
    )

    trial_agent = PPO(
        'MlpPolicy',
        train_env,
        learning_rate=cfg['learning_rate'],
        n_steps=cfg['n_steps'],
        batch_size=ppo_attn_cfg['batch_size'],
        n_epochs=ppo_attn_cfg['n_epochs'],
        gamma=ppo_attn_cfg['gamma'],
        gae_lambda=ppo_attn_cfg['gae_lambda'],
        clip_range=cfg['clip_range'],
        ent_coef=cfg['ent_coef'],
        vf_coef=ppo_attn_cfg['vf_coef'],
        policy_kwargs=policy_kwargs_hpo,
        device=device,
        verbose=0,
    )

    trial_agent.learn(total_timesteps=hpo_timesteps, progress_bar=False)

    val_eval = evaluate_episode(trial_agent, val_env, deterministic=True)
    val_reward = float(val_eval['reward'])
    print(f"Validation reward: {val_reward:.4f}")

    row = {
        **cfg,
        'trial': trial_idx,
        'val_reward': val_reward,
        'val_sharpe': float(val_eval.get('sharpe_ratio', 0.0)),
        'val_max_drawdown': float(val_eval.get('max_drawdown', 0.0)),
    }
    hpo_rows.append(row)

    if val_reward > best_score:
        best_score = val_reward
        best_cfg = cfg
        best_agent = trial_agent

ppo_attn_hpo_results = pd.DataFrame(hpo_rows).sort_values('val_reward', ascending=False)
print("\nTop configurations:")
print(ppo_attn_hpo_results.head())

if best_agent is None or best_cfg is None:
    raise RuntimeError("HPO failed to produce a valid best configuration.")

print(f"\nBest config: {best_cfg}")
print(f"Best validation reward: {best_score:.4f}")

ppo_attn_hpo_best = best_agent

test_eval_best = evaluate_episode(ppo_attn_hpo_best, test_env, deterministic=True)
print("\nBest HPO model test performance:")
for name in ['reward', 'cumulative_return', 'sharpe_ratio', 'max_drawdown']:
    value = float(test_eval_best.get(name, 0.0))
    print(f"  {name}: {value:.4f}")

fig_hpo = px.scatter(
    ppo_attn_hpo_results,
    x='trial',
    y='val_reward',
    color='learning_rate',
    symbol='n_steps',
    size='ent_coef',
    hover_data=['clip_range', 'val_sharpe', 'val_max_drawdown'],
    title='PPO Native-Attention HPO: Validation Reward by Trial',
    labels={'val_reward': 'Validation reward', 'trial': 'Trial #'},
)
fig_hpo.update_layout(height=500, width=1100)
fig_hpo.show()

print("HPO complete. Best model is stored in variable: ppo_attn_hpo_best")

=== Hyperparameter Fine-Tuning: PPO Native Attention ===


RuntimeError: Run Section 6.3 first so TemporalAttentionExtractor is defined.

## 7. Visualize Training Progress

In [ ]:
# Plot training progress with validation rewards (Plotly)
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if 'val_history' not in globals() or not val_history:
    print('No validation history found. Run training/evaluation cells that create val_history first.')
else:
    val_steps = [h['step'] for h in val_history]
    val_rewards = [h['reward'] for h in val_history]
    reward_improvement = [val_rewards[0]] + val_rewards
    best_idx = int(np.argmax(val_rewards))

    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=(
            'Validation Rewards During Training',
            'Reward Progression',
            'Validation Rewards (Best Performance)',
            'Training Summary',
        ),
        specs=[[{'type': 'xy'}, {'type': 'xy'}], [{'type': 'xy'}, {'type': 'table'}]],
        vertical_spacing=0.12,
    )

    fig.add_trace(
        go.Scatter(x=val_steps, y=val_rewards, mode='lines+markers', name='Validation Reward'),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter(x=list(range(len(reward_improvement))), y=reward_improvement, mode='lines+markers',
                   name='Reward Progression', line=dict(color='orange')),
        row=1,
        col=2,
    )

    fig.add_trace(
        go.Scatter(x=list(range(len(val_rewards))), y=val_rewards, mode='lines+markers',
                   name='Val Rewards (Best)', showlegend=False),
        row=2,
        col=1,
    )
    fig.add_hline(y=val_rewards[best_idx], line_dash='dash', line_color='green', row=2, col=1)

    summary_rows = [
        ['Total timesteps', f"{training_results.get('total_timesteps', 0):,}"],
        ['Validations run', f"{len(val_history)}"],
        ['Best val reward', f"{training_results.get('best_val_reward', max(val_rewards)):.4f}"],
        ['Final val reward', f"{val_rewards[-1]:.4f}"],
        ['Learning rate', str(dqn_cfg.get('learning_rate'))],
        ['Gamma', str(dqn_cfg.get('gamma'))],
        ['Buffer size', str(dqn_cfg.get('buffer_size'))],
        ['Batch size', str(dqn_cfg.get('batch_size'))],
    ]

    fig.add_trace(
        go.Table(
            header=dict(values=['Metric', 'Value']),
            cells=dict(values=[[r[0] for r in summary_rows], [r[1] for r in summary_rows]]),
        ),
        row=2,
        col=2,
    )

    fig.update_xaxes(title_text='Timesteps', row=1, col=1)
    fig.update_yaxes(title_text='Reward', row=1, col=1)
    fig.update_xaxes(title_text='Validation Checkpoint', row=1, col=2)
    fig.update_yaxes(title_text='Reward', row=1, col=2)
    fig.update_xaxes(title_text='Validation Checkpoint', row=2, col=1)
    fig.update_yaxes(title_text='Reward', row=2, col=1)
    fig.update_layout(height=900, width=1400, legend=dict(orientation='h', y=1.02, x=0))
    fig.show()

    print('Training Statistics:')
    print(f"  Validation checkpoints: {len(val_history)}")
    print(f"  Best validation reward: {max(val_rewards):.4f}")
    print(f"  Final validation reward: {val_rewards[-1]:.4f}")
    print(f"  Reward improvement: {val_rewards[-1] - val_rewards[0]:.4f}")

## 9.3 PPO + Attention Explainability (Surrogate + Saliency)

In [ ]:
# PPO-specific attention-style explainability via surrogate attention + policy saliency
print("=== PPO Attention + Saliency (Finance) ===")

import torch.nn as nn
import torch.optim as optim
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if 'ppo_agent' not in globals():
    raise RuntimeError("Please run the PPO training cell in Section 6.2 first.")

# Collect PPO rollout states from test period
obs, _ = test_env.reset()
ppo_states = []
n_collect_ppo = 220

for _ in range(n_collect_ppo):
    ppo_states.append(obs.copy())
    act, _ = ppo_agent.predict(obs, deterministic=True)
    act_id = int(act.item() if isinstance(act, np.ndarray) else act)
    obs, _, terminated, truncated, _ = test_env.step(act_id)
    if terminated or truncated:
        obs, _ = test_env.reset()

X_ppo_np = np.array(ppo_states, dtype=np.float32)  # [N, T, F]
X_ppo = torch.tensor(X_ppo_np, dtype=torch.float32, device=ppo_agent.device)

# Teacher targets from PPO policy distribution
with torch.no_grad():
    teacher_dist = ppo_agent.policy.get_distribution(X_ppo)
    teacher_probs = teacher_dist.distribution.probs.detach()  # [N, n_actions]


class AttentionPolicySurrogate(nn.Module):
    def __init__(self, in_dim, n_actions, d_model=64, n_heads=4):
        super().__init__()
        self.proj = nn.Linear(in_dim, d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.query = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.head = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, n_actions),
        )

    def forward(self, x):
        h = torch.relu(self.proj(x))
        h_attn, attn_w = self.attn(h, h, h, need_weights=True, average_attn_weights=False)
        q = self.query.expand(h_attn.size(0), -1, -1)
        pooled, pool_w = self.attn(q, h_attn, h_attn, need_weights=True, average_attn_weights=False)
        logits = self.head(pooled.squeeze(1))
        return logits, attn_w, pool_w


n_actions_ppo = train_env.action_space.n
ppo_surrogate = AttentionPolicySurrogate(
    in_dim=X_ppo.shape[-1],
    n_actions=n_actions_ppo,
    d_model=64,
    n_heads=4,
).to(ppo_agent.device)

optimizer = optim.Adam(ppo_surrogate.parameters(), lr=1e-3)
loss_fn = nn.KLDivLoss(reduction='batchmean')

batch_size = 64
epochs = 25
num_samples = X_ppo.size(0)

for epoch in range(epochs):
    perm = torch.randperm(num_samples, device=ppo_agent.device)
    epoch_loss = 0.0
    for i in range(0, num_samples, batch_size):
        idx = perm[i:i + batch_size]
        xb = X_ppo[idx]
        yb = teacher_probs[idx]

        logits, _, _ = ppo_surrogate(xb)
        log_probs = torch.log_softmax(logits, dim=1)
        loss = loss_fn(log_probs, yb)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        epoch_loss += float(loss.item()) * xb.size(0)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs} - KL: {epoch_loss / num_samples:.6f}")

# Extract attention weights
with torch.no_grad():
    _, _, ppo_pool_w = ppo_surrogate(X_ppo)

ppo_temporal_attn = ppo_pool_w.mean(dim=(0, 1, 2)).detach().cpu().numpy()
ppo_temporal_attn = ppo_temporal_attn / (ppo_temporal_attn.sum() + 1e-12)
ppo_attn_per_sample = ppo_pool_w.mean(dim=1).squeeze(1).detach().cpu().numpy()

# Compute PPO policy saliency wrt chosen action probability
n_sal = min(120, X_ppo_np.shape[0])
ppo_saliency_timestep = []

for i in range(n_sal):
    obs_leaf = torch.tensor(X_ppo_np[i], dtype=torch.float32, device=ppo_agent.device, requires_grad=True)
    dist = ppo_agent.policy.get_distribution(obs_leaf.unsqueeze(0))
    probs = dist.distribution.probs[0]
    chosen = int(torch.argmax(probs).item())
    score = torch.log(probs[chosen] + 1e-12)

    ppo_agent.policy.zero_grad(set_to_none=True)
    score.backward()

    grad_abs = obs_leaf.grad.detach().abs().cpu().numpy()  # [T, F]
    sal_t = grad_abs.sum(axis=1)
    sal_t = sal_t / (sal_t.sum() + 1e-12)
    ppo_saliency_timestep.append(sal_t)

ppo_saliency_timestep = np.array(ppo_saliency_timestep)
ppo_saliency_mean = ppo_saliency_timestep.mean(axis=0)

token_labels = [f"t-{len(ppo_temporal_attn)-1-i}" for i in range(len(ppo_temporal_attn))]

# Align returns for finance overlay
if 'ppo_prob_df' in globals() and 'actions_df' in globals():
    n_show = min(100, len(ppo_attn_per_sample), len(actions_df), len(ppo_prob_df))
    ret_series = actions_df['return'].iloc[:n_show].astype(float).values
else:
    n_show = min(100, len(ppo_attn_per_sample))
    ret_series = np.zeros(n_show, dtype=float)

fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.32, 0.36, 0.32],
    subplot_titles=(
        "PPO Temporal Importance: Surrogate Attention vs Policy Saliency",
        "PPO Attention Heatmap by Step",
        "Weekly Return vs PPO Focus on Recent Token",
    ),
)

fig.add_trace(
    go.Scatter(
        x=token_labels,
        y=ppo_temporal_attn,
        mode='lines+markers',
        name='PPO attention',
        line=dict(color='#0f766e', width=3),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=token_labels,
        y=ppo_saliency_mean,
        mode='lines+markers',
        name='PPO saliency',
        line=dict(color='#b45309', width=3, dash='dot'),
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Heatmap(
        z=ppo_attn_per_sample[:n_show].T,
        x=list(range(n_show)),
        y=token_labels,
        colorscale='Teal',
        colorbar=dict(title='attn', x=1.02),
        hovertemplate='step=%{x}<br>token=%{y}<br>attn=%{z:.3f}<extra></extra>',
        name='PPO attention heatmap',
    ),
    row=2,
    col=1,
)

fig.add_trace(
    go.Bar(
        x=list(range(n_show)),
        y=ret_series,
        name='Weekly return',
        marker_color='#94a3b8',
        opacity=0.5,
    ),
    row=3,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=list(range(n_show)),
        y=ppo_attn_per_sample[:n_show, -1],
        mode='lines',
        name='Attention on most recent token',
        line=dict(color='#7c2d12', width=2),
    ),
    row=3,
    col=1,
)

fig.update_layout(
    height=980,
    width=1400,
    title='PPO + Attention Explainability Dashboard',
    legend=dict(orientation='h', y=1.03, x=0),
)
fig.update_yaxes(title_text='importance', row=1, col=1)
fig.update_yaxes(title_text='token', row=2, col=1)
fig.update_yaxes(title_text='return / focus', row=3, col=1)
fig.update_xaxes(title_text='test step', row=3, col=1)
fig.show()

print("PPO attention diagnostics ready.")
print("Run order: Section 6.2 (PPO training) -> this section.")